# Chapitre 18 - Problèmes en grande dimension ($p \gg N$)
## Cours 2 : régularisation quadratique, astuce du noyau, pénalités L1, classification sans variables

Plan (sections 18.3 à 18.5 du livre) :

1. classifieurs linéaires à **pénalité quadratique** (RDA, logistique ridge, SVM) ;
2. l'**astuce de calcul** : quand $p > N$, tout se ramène à un problème en dimension
   $N$ grâce à la SVD (coût $O(N^2p)$ au lieu de $O(p^3)$) ;
3. pénalités **L1** : lasso, elastic net, *fused lasso* ;
4. classification quand on ne dispose que de **produits scalaires** ou de
   **distances** (noyaux de chaînes, centroïdes, ACP à noyau).

In [1]:
import time
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl import datasets
from esl import ch18_grande_dimension as g

## 1. Classifieurs linéaires à pénalité quadratique

### 1.1 Analyse discriminante régularisée (RDA)

La LDA utilise l'inverse de la matrice de covariance intra-classe $\hat\Sigma$
($p\times p$). Quand $p \gg N$, $\hat\Sigma$ est de rang au plus $N - K$ : elle est
singulière. La RDA la remplace par un mélange avec sa diagonale (éq. (18.9) du livre) :
$$\hat\Sigma(\gamma) = \gamma\,\hat\Sigma + (1-\gamma)\operatorname{diag}(\hat\Sigma), \qquad \gamma\in[0,1].$$
Pour $\gamma < 1$, $\hat\Sigma(\gamma)$ est définie positive (somme d'une matrice
positive et d'une matrice diagonale à termes $> 0$), donc inversible.
$\gamma = 0$ redonne la **LDA diagonale** du cours 1 ; $\gamma \to 1$ la LDA complète.

> **Intuition.** C'est l'analogue de la ridge : on « tire » la covariance estimée
> vers une matrice diagonale, ce qui stabilise son inversion. L'exercice 18.6 montre
> que le calcul se fait, lui aussi, en dimension $N$.

### 1.2 Régression logistique multinomiale ridge

Avec $K$ classes, on utilise la forme **symétrique** du modèle logistique
(éq. (18.10)) :
$$\Pr(G = k\mid X = x) = \frac{\exp(\beta_{k0} + x^T\beta_k)}{\sum_{\ell=1}^K \exp(\beta_{\ell 0} + x^T\beta_\ell)},$$
avec un vecteur $\beta_k\in\mathbb R^p$ **par classe**. Sans pénalité ce modèle est
sur-paramétré (ajouter le même vecteur à tous les $\beta_k$ ne change pas les
probabilités). On maximise la log-vraisemblance pénalisée (éq. (18.11)) :
$$\max_{\{\beta_{k0},\beta_k\}}\ \sum_{i=1}^N \log\Pr(g_i\mid x_i) - \frac\lambda2\sum_{k=1}^K\|\beta_k\|_2^2 .$$
La pénalité lève l'indétermination et impose $\sum_k\hat\beta_{kj} = 0$ pour chaque
$j$ (exercice 18.3). Le problème est **concave** (log-vraisemblance concave moins
une fonction convexe) : un algorithme de type Newton ou quasi-Newton le résout.